In [235]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"
# import threading
mt5.initialize()


True

In [236]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [237]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

13.03

In [238]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [240]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M1':mt5.TIMEFRAME_M1, 'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['ema1'] =rates_frame['close'].ewm(span=9, adjust=False).mean()
    rates_frame['ema2'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
    rates_frame['ema3'] =rates_frame['close'].ewm(span=50, adjust=False).mean()

#     rates_frame['ema'] =ema(rates_frame['close'], 9)
#     rates_frame['ema'] =ema(rates_frame['close'], 9)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)
    
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
#         print(rates_frame.head())
    # Calculate Supertren
    return rates_frame

In [241]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [242]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [243]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
b = get_values(symbol, 2500, 2, 'M30')
a = get_values(symbol, 5000, 2, 'M5')

timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
print(b.iloc[0].name)
print(a.iloc[0].name)

for i in range(4, len(a)-1):
    try:
        start_index = b.index.get_loc(a.iloc[i].name)
        break
    except:
        pass


for j in range(start_index, len(b)-1):
    if b.iloc[j].name.weekday() !=5 and b.iloc[j].close < b.iloc[j].ema3 and b.iloc[j].open > b.iloc[j].ema3 and check ==0:
        try:
            index = a.index.get_loc(b.iloc[j].name)
            index_end = a.index.get_loc(b.iloc[j+1].name)
        except:
            index = a.index.get_loc(b.iloc[j].name + pd.Timedelta(minutes=5))
            index_end = a.index.get_loc(b.iloc[j+1].name + pd.Timedelta(minutes=5))
        print("++++++"*20)
            
        for i in range(index, index_end):
#              i)
            if check== 0:
                if a.iloc[i].name.weekday() !=5 and a.iloc[i].close < a.iloc[i].ema1 and a.iloc[i].close < a.iloc[i].ema2:
                    print("=="*20)
                    print(index, index_end)
                    

                    print(f"SELL  {a.iloc[i].name} -- {b.iloc[j].name} -- {a.iloc[index].name} -- {a.iloc[index_end].name}")
                    buy_price = a.iloc[i].close
                    check=1
                    
            elif check==1:
                sell_price = a.iloc[i].close
                pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10)
                print(f"{pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
                if pp > 25:
        #             pp1 = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10) 
                    print(f"PP {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
                    profit.append(pp)
                    check = 0
#                     break
                elif pp< -10:
                    print(f"LESS PP {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
                    profit.append(pp)
                    check = 0
#                     break
        if check !=0:
            pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10)
            print(f"LAST  {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check=0
#             break
            
#     if b.iloc[j].name.weekday() !=5 and b.iloc[j].close > b.iloc[j].ema3 and b.iloc[j].open < b.iloc[j].ema3 and check ==0:
#         try:
#             index = a.index.get_loc(b.iloc[j].name)
#             index_end = a.index.get_loc(b.iloc[j+1].name)
#         except:
#             index = a.index.get_loc(b.iloc[j].name + pd.Timedelta(minutes=5))
#             index_end = a.index.get_loc(b.iloc[j+1].name + pd.Timedelta(minutes=5))
            
#         for i in range(index, index_end):
#             if check== 0:
#                 if a.iloc[i].name.weekday() !=5 and a.iloc[i].close < a.iloc[i].ema1 and a.iloc[i].close < a.iloc[i].ema2 and \
#                 (a.iloc[i].open > a.iloc[i].ema1 or a.iloc[i].open > a.iloc[i].ema2):
#                     print("=="*20)

#                     print(f"BUY  {a.iloc[i].name} -- {b.iloc[j].name}-- {index} -- {index_end}")
#                     buy_price = a.iloc[i].close
#                     check=2
                    
#             elif check==2:
#                 sell_price = a.iloc[i].close
#                 pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - (2.50*0.1*10)
#                 print(f"{pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 if pp > 25:
#         #             pp1 = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - (2.50*0.1*10) 
#                     print(f"PP {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     profit.append(pp)
#                     check = 0
# #                     break
#                 elif pp< -10:
#                     print(f"PP Less {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     profit.append(pp)
#                     check = 0
# #                     break
#         if check !=0:
#             pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - (2.50*0.1*10)
#             print(f"LAST  {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
#             check=0


#             break

#             elif check==2:
#                 sell_price = a.iloc[i].close
#                 pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
#                 print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                 if (sell_price-buy_price) > 0.920:
#                     pp1 = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - (2.50*0.1*10) 
#                     print(f"PP {pp1}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     profit.append(pp1)
#                     check = 0
#                 if (buy_price - sell_price) > 0.100:
#                     pp1 = price_action(symbol, 0.1, buy_price, buy_price-0.100, mt5.ORDER_TYPE_BUY) -(2.50*0.1*10)
#                     profit.append(pp)
#                     print(f"PP_LOSS {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     check = 0
#                 elif a.iloc[i].rsi1 < 15:
#                     profit.append(pp)
#                     print(f"RSI {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     check =0

#                 elif a.iloc[i].rsi1 > 90:
#                     profit.append(pp)
#                     print(f"RSI_DOne {pp}--{ a.iloc[i].rsi1}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#                     check =0

2024-08-13 02:00:00
2024-09-17 06:05:00
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
358 364
SELL  2024-09-18 12:00:00 -- 2024-09-18 12:00:00 -- 2024-09-18 12:00:00 -- 2024-09-18 12:30:00
4.62--22.844864379611465---60028.99--60100.17--2024-09-18 12:05:00
6.869999999999999--21.198684148229773---60006.45--60100.17--2024-09-18 12:10:00
10.81--18.48201130074581---59967.04--60100.17--2024-09-18 12:15:00
4.46--34.32159025308857---60030.61--60100.17--2024-09-18 12:20:00
12.17--26.91939209813951---59953.5--60100.17--2024-09-18 12:25:00
LAST  12.17--26.91939209813951---59953.5--60100.17--2024-09-18 12:25:00
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
388 394
SELL  2024-09-18 14:50:00 -- 2024-09-18 14:30:00 -- 2024-09-18 14:30:00 -- 2024-09-18 15:00:00
-0.78--39.17537523319832---59842.81--59860.03--2024-09-18 14:55:00
LAST  -0.78--39.17537523319

SELL  2024-09-29 08:30:00 -- 2024-09-29 08:30:00 -- 2024-09-29 08:30:00 -- 2024-09-29 09:00:00
-3.04--19.72473553463591---65595.23--65589.82--2024-09-29 08:35:00
-0.55--17.26660200271523---65570.28--65589.82--2024-09-29 08:40:00
1.25--15.634830701792893---65552.37--65589.82--2024-09-29 08:45:00
-3.32--34.14243038877461---65598.02--65589.82--2024-09-29 08:50:00
-4.41--37.93198901972224---65608.91--65589.82--2024-09-29 08:55:00
LAST  -4.41--37.93198901972224---65608.91--65589.82--2024-09-29 08:55:00
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
3499 3505
SELL  2024-09-29 15:50:00 -- 2024-09-29 15:30:00 -- 2024-09-29 15:30:00 -- 2024-09-29 16:00:00
2.13--31.097219924972407---65617.17--65663.49--2024-09-29 15:55:00
LAST  2.13--31.097219924972407---65617.17--65663.49--2024-09-29 15:55:00
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
3529 3535


In [244]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

268.88000000000005
Total negative sm -->-180.36999999999995
Total negative -->19
Total positive sm -->449.25
Total positive -->23
Length 42


In [113]:
profit.sort()

In [114]:
profit

[-20.47,
 -15.21,
 -13.11,
 -8.49,
 -4.73,
 -3.99,
 -2.72,
 -2.01,
 -1.43,
 -0.78,
 0.020000000000000018,
 1.71,
 1.8600000000000003,
 2.05,
 2.34,
 4.0600000000000005,
 5.28,
 5.38,
 5.8100000000000005,
 5.92,
 6.869999999999999,
 6.960000000000001,
 6.970000000000001,
 8.37,
 10.62,
 12.04,
 13.48,
 14.14,
 14.18,
 14.23,
 18.18,
 27.22,
 27.33,
 29.39]

In [224]:
for i in range(4988,4994):
    print(i)

4988
4989
4990
4991
4992
4993


In [229]:
a.iloc[4989].name

Timestamp('2024-09-22 22:05:00')